# Control genes for the p2 covariates (RUVg W)

RUVg negative controls must be unaffected by the biology of interest and affected by the unwanted variation
(Gagnon-Bartsch & Speed 2012, Biostatistics 13:539, doi:10.1093/biostatistics/kxr034; Risso et al. 2014,
Nat Biotechnol 32:896, doi:10.1038/nbt.2931). cfRNA is mostly of hematopoietic origin and disease shifts the cell-type
mix (Vorperian et al. 2022, Nat Biotechnol 40:855, doi:10.1038/s41587-021-01188-9), so a control gene must be measured
reliably in every sample and be insensitive to the blood cell-type composition:

| criterion | rule | source |
|---|---|---|
| C1 stable detection | count > 0 in >= `DETECT_MIN` of the samples of every batch x phenotype group (n >= `MIN_GROUP_N`), disease and HC alike | this data |
| C2a uniform across blood cell types | nCPM >= `HPA_MIN_NCPM` in every blood cell type of `BLOOD_TYPES` and specificity tau <= `TAU_MAX` across them | HPA 25.1 single cell type (Karlsson et al. 2021, Sci Adv 7:eabh2169, doi:10.1126/sciadv.abh2169); tau: Kryuchkova-Mostacci & Robinson-Rechavi 2017, Brief Bioinform 18:205, doi:10.1093/bib/bbw008 |
| C2b stable in whole blood across donors | normalised CPM >= 1 in >= `WB_DETECT_MIN` of GTEx v10 whole-blood donors and Gini < `GINI_MAX` across donors | GTEx v10; Gini < 0.2 "in circulation" for endogenous controls in Nesselbush et al. 2025, Nature 641:759, doi:10.1038/s41586-025-08834-1 |

Three C1 variants are compared end to end (`SETS`): per batch x phenotype group at 95% (`group95`) and 90% (`group90`),
and per batch (all samples of a batch pooled, disease and HC together) at 95% (`batch95`); C2a and C2b are the same for all.
`CHOSEN` is the one frozen for p2. Control set = C1 & C2a & C2b, plus the four chrY genes of the Staedtler et al. 2013 sex-marker panel (Biomarkers 18:436,
doi:10.3109/1354750X.2013.811538), which C1 removes (zero in females) and which carry the sex axis. Disease samples enter
only through C1 (detection); phenotype association is reported, never used to select. N follows from the criteria; this
notebook chooses k. W is fitted on HC only here; the p2 provider refits it in every CV fold / LOBO training set.
The h5ad `RUVg_Platelet_*` / `Proposed_Full_*` layers are not used.

In [ ]:
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import f as f_dist

parent_dir = str(Path.cwd().parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from viz_style import FS_ANNOT_SM, apply_style
apply_style()

import _FinalModelingMethodImprovement.config as config
from _FinalModelingMethodImprovement.engine.validation.lobo_engine import _decode_cat, load_full_data
from _FinalModelingMethodImprovement.eval.insilico_sweep import cell_path, fold_spec, universe

OUT = config.CONTROL_GENE_DIR
FIG_DIR = config.RESULTS_DIR / 'Figures'
OUT.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
CONTROL_LIST = config.SPLITS_DIR / 'control_genes_blood.txt'

DETECT_MIN = 0.95
MIN_GROUP_N = 5
SETS = {'group95': ('group', 0.95), 'group90': ('group', 0.90), 'batch95': ('batch', 0.95)}
SET_COLOR = {'group95': '#0072B2', 'group90': '#E69F00', 'batch95': '#009E73'}
CHOSEN = 'group95'
BLOOD_TYPES = ['platelets', 'erythrocytes', 'neutrophils', 'monocytes', 't-cells', 'b-cells', 'nk-cells', 'plasma cells', 'cdc', 'pdcs']
HPA_MIN_NCPM = 1.0
TAU_MAX = 0.3
WB_DETECT_MIN = 0.95
GINI_MAX = 0.2
SEX_GENES = ['RPS4Y1', 'EIF1AY', 'DDX3Y', 'KDM5D']
DEPTH = 'log(Total Reads)'
K_MAX = 10
K_MAIN = 6
STAB_B = 100
STAB_MIN = 0.95
pd.set_option('display.width', 250)

In [ ]:
data = load_full_data()
genes = list(data['gene_col'])
Y = data['Y'][:, [data['gene_col'][g] for g in genes]]
lib = data['Y'].sum(1)
del data['Y']
logcpm = np.log2(Y / lib[:, None] * 1e6 + 1).astype(np.float32)
col = {g: i for i, g in enumerate(genes)}
base_id = np.array([g.split('.')[0] for g in genes])
with h5py.File(config.H5AD_PATH, 'r') as f:
    sym = np.array([dict(zip(f['var/_index'][()].astype(str), _decode_cat(f['var/GeneName']))).get(g, '') for g in genes])
    chrom = dict(zip(f['var/_index'][()].astype(str), _decode_cat(f['var/Chromosome'])))
    obs_names = f['obs/_index'][()].astype(str)
    sex = pd.Series(_decode_cat(f['obs/sex']), index=obs_names).loc[data['names']].values
    sex_score = pd.Series(f['obs/Sex Score'][()], index=obs_names).loc[data['names']].values
batch, pheno = data['batch'], data['phenotype']
hc_fit = data['is_hc'] & ~np.isin(batch, list(data['small_hc_batches']))
modeled = set(pd.read_csv(config.ENGINE_MIXED_DIR / 'training_summary.csv', usecols=['gene'])['gene'])
sex_genes = [g for g in genes if sym[col[g]] in SEX_GENES and chrom.get(g) in ('chrY', 'Y')]
assert sorted(sym[col[g]] for g in sex_genes) == sorted(SEX_GENES)
print(f'samples {len(batch)} (HC used for W {hc_fit.sum()}), protein-coding genes {len(genes)}')

## 1. C1: stable detection in disease and HC

Detection = fraction of samples with count > 0, per batch x phenotype group with at least `MIN_GROUP_N` samples; a gene
passes when its lowest group is >= `DETECT_MIN`. For comparison the table also gives the per-batch rule (all samples of a
batch with >= 10 samples pooled), under which a small disease group can fail without the gene failing.

In [ ]:
grp = pd.Series([f'{b} | {p}' for b, p in zip(batch, pheno)])
groups = grp.value_counts().loc[lambda v: v >= MIN_GROUP_N].index
det_group = pd.DataFrame({g: (Y[(grp == g).values] > 0).mean(0) for g in groups}, index=genes)
big_batches = pd.Series(batch).value_counts().loc[lambda v: v >= 10].index
det_batch = np.min([(Y[batch == b] > 0).mean(0) for b in big_batches], axis=0)
min_group = det_group.min(1).values
c1 = min_group >= DETECT_MIN
print(f'groups {len(groups)} ({grp.isin(groups).sum()} samples, {pd.Series(pheno[grp.isin(groups).values]).nunique()} phenotypes)')
display(pd.DataFrame({t: dict(per_group=int((min_group >= t).sum()), per_batch=int((det_batch >= t).sum()))
                      for t in (0.8, 0.9, 0.95, 0.99)}).T.rename_axis('DETECT_MIN'))
worst = det_group[c1 | (det_batch >= DETECT_MIN)].idxmin(1).value_counts()
print('group that sets the minimum, genes passing either rule (top 10):')
display(worst.head(10))

## 2. C2a: uniform expression across blood cell types (HPA)

HPA 25.1 single cell type nCPM, restricted to the blood cell types `BLOOD_TYPES` (platelets, erythrocytes, neutrophils,
monocytes, T, B, NK, plasma cells, cDC, pDC). tau = sum(1 - x_i / max x) / (n - 1) on log2(nCPM + 1), 0 = uniform,
1 = one cell type only. A gene must be expressed (nCPM >= `HPA_MIN_NCPM`) in every one of them, so it is present in blood
and does not move with the share of any single cell type. Genes absent from HPA fail.

In [ ]:
with zipfile.ZipFile(config.HPA_SINGLE_CELL_ZIP) as z:
    hpa = pd.read_csv(z.open('rna_single_cell_type.tsv'), sep='\t')
hpa = hpa[hpa['Cell type'].isin(BLOOD_TYPES)].pivot_table(index='Gene', columns='Cell type', values='nCPM')
assert sorted(hpa.columns) == sorted(BLOOD_TYPES)
lx = np.log2(hpa + 1)
tau = ((1 - lx.div(lx.max(1), axis=0)).sum(1) / (lx.shape[1] - 1)).reindex(base_id).values
blood_all = (hpa >= HPA_MIN_NCPM).all(1).reindex(base_id).astype(float).fillna(0).values.astype(bool)
c2a = blood_all & (tau <= TAU_MAX)
print(f'protein-coding genes in HPA: {np.isfinite(tau).sum()}/{len(genes)}')
display(pd.DataFrame({t: dict(all_genes=int((blood_all & (tau <= t)).sum()), within_C1=int((c1 & blood_all & (tau <= t)).sum()))
                      for t in (0.2, 0.3, 0.4, 0.5)}).T.rename_axis('TAU_MAX'))

fig, ax = plt.subplots(figsize=(5, 3.5))
bins = np.linspace(0, 1, 41)
ax.hist(tau[blood_all], bins=bins, color='0.8', label='expressed in all blood types')
ax.hist(tau[blood_all & c1], bins=bins, histtype='step', color='#0072B2', lw=1.5, label='... and C1')
ax.axvline(TAU_MAX, color='k', ls=':')
ax.set_xlabel('tau across blood cell types (HPA)'); ax.set_ylabel('Genes')
ax.legend(frameon=False, fontsize=FS_ANNOT_SM)
fig.savefig(FIG_DIR / 'control_2a_hpa_tau.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. C2b: stable expression in whole blood across donors (GTEx v10)

GTEx v10 whole-blood read counts. Raw CPM is dominated by the donor-to-donor globin share, which moves every other gene
together, so expression is normalised by median-of-ratios size factors (genes with counts > 0 in every donor) and rescaled
to the CPM scale before the Gini coefficient across donors is taken (Nesselbush et al. used TMM before their Gini < 0.2
rule). Both versions are shown. Expressed = normalised CPM >= 1 in >= `WB_DETECT_MIN` of donors.

In [ ]:
def gini(m):
    s = np.sort(m, 1)
    n = s.shape[1]
    with np.errstate(invalid='ignore'):
        return ((2 * np.arange(1, n + 1) - n - 1) * s).sum(1) / (n * s.sum(1))


WB_CACHE = OUT / 'gtex_wb_gene_stats.csv'
if WB_CACHE.is_file():
    wbs = pd.read_csv(WB_CACHE, index_col=0)
else:
    wb = pd.read_csv(config.GTEX_WB_GCT, sep='\t', skiprows=2, index_col=0).drop(columns='Description')
    wb.index = wb.index.str.split('.').str[0]
    wb = wb[~wb.index.duplicated()].astype(float)
    cpm = wb / wb.sum() * 1e6
    lg = np.log(wb[(wb > 0).all(1)].values)
    sf = np.exp(np.median(lg - lg.mean(1, keepdims=True), 0))
    nx = wb / sf
    nx = nx * (cpm.values.mean() / nx.values.mean())
    wbs = pd.DataFrame(dict(detect=(nx >= 1).mean(1), gini_norm=gini(nx.values), gini_cpm=gini(cpm.values)), index=wb.index)
    wbs.to_csv(WB_CACHE)
    print(f'GTEx whole-blood donors: {wb.shape[1]}')
wb_detect = wbs['detect'].reindex(base_id).values
wb_gini = wbs['gini_norm'].reindex(base_id).values
c2b = (wb_detect >= WB_DETECT_MIN) & (wb_gini < GINI_MAX)
expressed = wb_detect >= WB_DETECT_MIN
display(pd.DataFrame({t: dict(all_genes=int((expressed & (wb_gini < t)).sum()), within_C1=int((c1 & expressed & (wb_gini < t)).sum()),
                              within_C1_C2a=int((c1 & c2a & expressed & (wb_gini < t)).sum()),
                              within_C1_raw_cpm_gini=int((c1 & expressed & (wbs['gini_cpm'].reindex(base_id).values < t)).sum()))
                      for t in (0.15, 0.2, 0.25, 0.3)}).T.rename_axis('GINI_MAX'))

fig, ax = plt.subplots(figsize=(4.5, 4))
m = c1 & expressed
ax.scatter(wbs['gini_cpm'].reindex(base_id).values[m], wb_gini[m], s=3, color='0.5')
ax.plot([0, 1], [0, 1], color='k', lw=0.6)
ax.axhline(GINI_MAX, color='k', ls=':')
ax.set_xlabel('Gini, raw CPM'); ax.set_ylabel('Gini, median-ratio normalised')
ax.set_title('C1 genes expressed in GTEx whole blood')
fig.savefig(FIG_DIR / 'control_2b_gtex_gini.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Candidate control sets

For each set in `SETS`: C1 variant & C2a & C2b, plus the sex genes. Descriptive checks, none used to select:
- PanglaoDB blood / immune markers in the set (a uniform gene should rarely be a cell-type marker)
- phenotype association: within-batch phenotype partial R^2 per gene (F-test of `logCPM ~ batch + phenotype` vs
  `logCPM ~ batch`, all samples), the set vs the other genes passing the same C1 variant

In [ ]:
def basis(X):
    u, s, _ = np.linalg.svd(X, full_matrices=False)
    return u[:, s > s[0] * 1e-10]


def rss(Q, Yc):
    return (Yc ** 2).sum(0) - ((Q.T @ Yc) ** 2).sum(0)


detect = {'group': min_group, 'batch': det_batch}
c1_of = {s: detect[u] >= t for s, (u, t) in SETS.items()}
controls = {}
for s, c1s in c1_of.items():
    sel = c1s & c2a & c2b
    controls[s] = [genes[i] for i in np.where(sel)[0]] + [g for g in sex_genes if not sel[col[g]]]
    (OUT / f'control_genes_{s}.txt').write_text('\n'.join(controls[s]))
cidx = {s: np.array([col[g] for g in c]) for s, c in controls.items()}
display(pd.DataFrame({s: dict(C1=int(c1s.sum()), C1_C2a=int((c1s & c2a).sum()), C1_C2b=int((c1s & c2b).sum()),
                              C1_C2a_C2b=int((c1s & c2a & c2b).sum()), with_sex=len(controls[s])) for s, c1s in c1_of.items()}).T)
display(pd.DataFrame({a: {b: len(set(controls[a]) & set(controls[b])) for b in SETS} for a in SETS}).rename_axis('shared genes'))

PH_CACHE = OUT / 'pheno_r2.csv'
ph_genes = sorted(set().union(*[np.where(c)[0] for c in c1_of.values()], *[set(i) for i in cidx.values()]))
ph = pd.read_csv(PH_CACHE, index_col='gene') if PH_CACHE.is_file() else None
if ph is None or not set(genes[i] for i in ph_genes) <= set(ph.index):
    B = pd.get_dummies(batch).values.astype(float)
    P = pd.get_dummies(pheno).values.astype(float)
    Q0, Q1 = basis(B), basis(np.hstack([B, P]))
    df1, df2 = Q1.shape[1] - Q0.shape[1], len(batch) - Q1.shape[1]
    L = logcpm[:, ph_genes].astype(np.float64)
    r0, r1 = rss(Q0, L), rss(Q1, L)
    F = ((r0 - r1) / df1) / (r1 / df2)
    ph = pd.DataFrame(dict(pheno_r2=(r0 - r1) / r0, p=f_dist.sf(F, df1, df2)), index=pd.Index([genes[i] for i in ph_genes], name='gene'))
    ph.to_csv(PH_CACHE)

pg = pd.read_csv(config.PANGLAO_TSV, sep='\t')
pg = pg[pg['species'].str.contains('Hs') & (pg['organ'].isin(['Blood', 'Immune system']) | (pg['cell type'] == 'Hematopoietic stem cells'))]
marker_of = pg.groupby('official gene symbol')['cell type'].agg(lambda x: ', '.join(sorted(set(x))))
tables, others = {}, {}
for s, ci in cidx.items():
    t = pd.DataFrame(dict(symbol=sym[ci], min_group_detect=min_group[ci], min_batch_detect=det_batch[ci], tau=tau[ci], wb_gini=wb_gini[ci],
                          mean_logcpm=logcpm[:, ci].mean(0), panglao_blood_marker=pd.Series(sym[ci]).map(marker_of).fillna('').values),
                     index=pd.Index(controls[s], name='gene')).join(ph)
    t.to_csv(OUT / f'control_genes_table_{s}.csv')
    tables[s] = t
    others[s] = ph.loc[[genes[i] for i in np.where(c1_of[s])[0] if genes[i] not in set(controls[s])], 'pheno_r2']
display(pd.DataFrame({s: dict(n=len(t), panglao_blood_markers=int((t['panglao_blood_marker'] != '').sum()),
                              pheno_r2_median=t['pheno_r2'].median(), pheno_r2_p90=t['pheno_r2'].quantile(0.9),
                              other_C1_pheno_r2_median=others[s].median(), other_C1_pheno_r2_p90=others[s].quantile(0.9),
                              mean_logcpm_median=t['mean_logcpm'].median()) for s, t in tables.items()}).T.round(4))
for s, t in tables.items():
    print(f"{s} PanglaoDB markers: " + ', '.join(f"{r.symbol} ({r.panglao_blood_marker})" for r in t[t['panglao_blood_marker'] != ''].itertuples()))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for s, t in tables.items():
    kw = dict(histtype='step', lw=1.5, density=True, color=SET_COLOR[s])
    axes[0].hist(t['pheno_r2'], bins=25, label=f'{s} (n={len(t)})', **kw)
    axes[0].hist(others[s], bins=40, ls=':', **kw)
    axes[2].hist(t['mean_logcpm'], bins=20, label=s, **kw)
for s in reversed(list(SETS)):
    axes[1].scatter(tables[s]['tau'], tables[s]['wb_gini'], s=6, color=SET_COLOR[s], label=s)
axes[0].set_xlabel('Within-batch phenotype partial R2'); axes[0].set_ylabel('Density')
axes[0].set_title('solid: set, dotted: other C1 genes')
axes[1].axvline(TAU_MAX, color='k', ls=':'); axes[1].axhline(GINI_MAX, color='k', ls=':')
axes[1].set_xlabel('tau across blood cell types (HPA)'); axes[1].set_ylabel('Gini across GTEx WB donors')
axes[2].set_xlabel('Mean log2 CPM (all samples)'); axes[2].set_ylabel('Density')
for ax in axes:
    ax.legend(frameon=False, fontsize=FS_ANNOT_SM, markerscale=3)
fig.tight_layout()
fig.savefig(FIG_DIR / 'control_4_sets.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. RUVg factors per set and the choice of k

`RUVSeq::RUVg(isLog=TRUE, center=TRUE)` on the HC control matrix (log2 CPM, library size over all genes, as
`engine/core/covariates.py`); other samples are projected, W_new = (x - HC mean) V D^-1. Per set and k: share of
control-gene variance, bootstrap stability (mean canonical correlation between W_1..k of the full HC fit and of a fit on a
bootstrap resample, both evaluated on all HC; 5th percentile over `STAB_B` resamples, the same resamples for every set),
R^2 of known sex (known-sex HC) and of obs['Sex Score'] (all HC) on W_1..k. The R check compares Python and R W (|r|).

In [ ]:
def ruvg_fit(x, k):
    m = x.mean(0)
    _, d, vt = np.linalg.svd(x - m, full_matrices=False)
    return dict(mean=m, v=vt[:k].T, d=d[:k], var_frac=d ** 2 / (d ** 2).sum())


def ruvg_transform(fit, x):
    return (x - fit['mean']) @ fit['v'] / fit['d']


def canon_corr(a, b):
    return np.linalg.svd(basis(a - a.mean(0)).T @ basis(b - b.mean(0)), compute_uv=False)


def r2_on(W_, y):
    y = y - y.mean()
    return 1 - rss(basis(np.column_stack([np.ones(len(y)), W_])), y[:, None])[0] / (y ** 2).sum()


hc_rows = np.where(hc_fit)[0]
rng = np.random.default_rng(0)
boots = [rng.choice(hc_rows, len(hc_rows)) for _ in range(STAB_B)]
KS = list(range(1, K_MAX + 1))
male, female = hc_fit & (sex == 'male'), hc_fit & (sex == 'female')
known = male | female
W, kts, stabs = {}, {}, {}
for s, ci in cidx.items():
    x_hc = logcpm[hc_rows][:, ci].astype(np.float64)
    fit = ruvg_fit(x_hc, K_MAX)
    W[s] = ruvg_transform(fit, logcpm[:, ci].astype(np.float64))
    pd.DataFrame(W[s], index=data['names'], columns=[f'W_{i + 1}' for i in range(K_MAX)]).assign(
        hc_fit=hc_fit, batch=batch, phenotype=pheno).to_csv(OUT / f'W_{s}.csv')
    with tempfile.TemporaryDirectory() as tmp:
        pd.DataFrame(x_hc.T).to_csv(f'{tmp}/x.csv', index=False)
        r = ('suppressMessages(library(RUVSeq)); x <- as.matrix(read.csv(commandArgs(TRUE)[1])); '
             f'w <- RUVg(x, seq_len(nrow(x)), k = {K_MAX}, isLog = TRUE, center = TRUE)$W; '
             'write.csv(w, commandArgs(TRUE)[2], row.names = FALSE)')
        subprocess.run([str(config.RUVSEQ_RSCRIPT), '-e', r, f'{tmp}/x.csv', f'{tmp}/w.csv'], check=True)
        w_r = pd.read_csv(f'{tmp}/w.csv').values
    r_min = min(abs(np.corrcoef(W[s][hc_rows][:, i], w_r[:, i])[0, 1]) for i in range(K_MAX))
    assert r_min > 0.9999, (s, r_min)
    ref = ruvg_transform(fit, x_hc)
    st = {k: [] for k in KS}
    for rb in boots:
        wb_ = ruvg_transform(ruvg_fit(logcpm[rb][:, ci].astype(np.float64), K_MAX), x_hc)
        for k in KS:
            st[k].append(canon_corr(ref[:, :k], wb_[:, :k]).mean())
    stabs[s] = {k: np.array(v) for k, v in st.items()}
    kts[s] = pd.DataFrame({k: dict(var_frac_cum=fit['var_frac'][:k].sum(), var_frac_k=fit['var_frac'][k - 1],
                                   stab_p5=np.quantile(stabs[s][k], 0.05), stab_median=np.median(stabs[s][k]),
                                   sex_r2=r2_on(W[s][known][:, :k], male[known].astype(float)),
                                   sex_score_r2=r2_on(W[s][hc_rows][:, :k], sex_score[hc_rows])) for k in KS}).T.rename_axis('k')
    print(f'{s}: R check min |r| = {r_min:.6f}')
kt = pd.concat(kts, names=['set'])
kt.to_csv(OUT / 'k_table.csv')
display(kt[['var_frac_cum', 'stab_p5', 'sex_r2', 'sex_score_r2']].unstack('set').round(4))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for s in SETS:
    d = kts[s]
    axes[0].plot(KS, 100 * d['var_frac_cum'], 'o-', ms=3, color=SET_COLOR[s], label=s)
    axes[1].plot(KS, d['stab_p5'], 'o-', ms=3, color=SET_COLOR[s], label=s)
    axes[2].plot(KS, d['sex_r2'], 'o-', ms=3, color=SET_COLOR[s], label=f'{s}, known sex')
    axes[2].plot(KS, d['sex_score_r2'], 's--', ms=3, color=SET_COLOR[s], label=f'{s}, Sex Score')
axes[0].set_ylabel('Cumulative control-gene variance (%)')
axes[1].axhline(STAB_MIN, color='k', ls=':'); axes[1].set_ylabel(f'Bootstrap stability, 5th pct (B = {STAB_B})')
axes[2].set_ylabel('R2 on W_1..k (HC)')
for ax in axes:
    ax.set_xlabel('k'); ax.axvline(K_MAIN, color='0.5', ls=':', lw=0.8)
    ax.legend(frameon=False, fontsize=FS_ANNOT_SM)
fig.tight_layout()
fig.savefig(FIG_DIR / 'control_5_k_scree_stability_sex.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Explained HC variance and collinearity: depth + W_1..k vs BIAS10

Y = HC log2 CPM of the genes the engine models, minus the union of all candidate control genes (same Y for every set).
Multivariate adjusted R^2 of depth + W_1..k (the p2 covariates) against BIAS10 (p0); `batch_left` =
R^2(covariates + Batch_ID) - R^2(covariates), the batch variance the covariates leave. VIF = diag of the inverse
correlation matrix (HC).

In [ ]:
def adj_r2(Yc, X):
    Q = basis(np.column_stack([np.ones(len(Yc)), X]))
    r2 = 1 - rss(Q, Yc).sum() / (Yc ** 2).sum()
    n, p = len(Yc), Q.shape[1] - 1
    return max(0.0, 1 - (1 - r2) * (n - 1) / (n - p - 1))


def vif(X):
    return np.diag(np.linalg.inv(np.corrcoef(X.T)))


excl = set().union(*controls.values())
Yc = logcpm[hc_fit][:, [col[g] for g in genes if g in modeled and g not in excl]].astype(np.float64)
Yc -= Yc.mean(0)
Xb = data['X_raw'][hc_fit]
Xb = (Xb - Xb.mean(0)) / Xb.std(0)
dz = Xb[:, [config.BIAS_COLUMNS.index(DEPTH)]]
Bt = pd.get_dummies(batch[hc_fit]).values.astype(float)
ref_bias = dict(adj_r2=adj_r2(Yc, Xb), batch_left=adj_r2(Yc, np.hstack([Xb, Bt])) - adj_r2(Yc, Xb), max_vif=vif(Xb).max())
rows = []
for s in SETS:
    Wh = W[s][hc_fit]
    for k in KS:
        X = np.hstack([dz, Wh[:, :k]])
        a = adj_r2(Yc, X)
        rows.append(dict(set=s, k=k, adj_r2=a, batch_left=adj_r2(Yc, np.hstack([X, Bt])) - a, max_vif=vif(X).max()))
rda = pd.DataFrame(rows)
rda.to_csv(OUT / 'rda_vif.csv', index=False)
print(f"BIAS10: adjusted R2 {ref_bias['adj_r2']:.4f}, batch left {ref_bias['batch_left']:.4f}, max VIF {ref_bias['max_vif']:.2f}")
display(rda.pivot(index='k', columns='set', values=['adj_r2', 'batch_left', 'max_vif']).round(4))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for s in SETS:
    d = rda[rda['set'] == s]
    for ax, m in zip(axes, ('adj_r2', 'batch_left', 'max_vif')):
        ax.plot(d['k'], (100 if m != 'max_vif' else 1) * d[m], 'o-', ms=3, color=SET_COLOR[s], label=s)
for ax, m, lab in zip(axes, ('adj_r2', 'batch_left', 'max_vif'),
                      ('HC variance explained (adjusted R2, %)', 'Batch variance left (%)', 'Max VIF')):
    ax.axhline((100 if m != 'max_vif' else 1) * ref_bias[m], color='k', ls=':', label='BIAS10')
    ax.set_xlabel('k (depth + W_1..k)'); ax.set_ylabel(lab); ax.axvline(K_MAIN, color='0.5', ls=':', lw=0.8)
    ax.legend(frameon=False, fontsize=FS_ANNOT_SM)
axes[2].set_yscale('log')
fig.tight_layout()
fig.savefig(FIG_DIR / 'control_6_variance_vif.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. In-silico projection absorption

Does W take up injected signal? Frozen in-silico cells (`eval/insilico_sweep.py`: 500 random universe genes per fold,
every held-out HC of each LOBO batch / CV fold). Per fold, W is fitted on the training HC and projected onto the injected
counts (library size moves with the injection). Linear proxy of the engine: per gene OLS of log2 CPM on [1, depth, W_1..k]
fitted on the training HC.
- `absorbed`: share of the injected log2 CPM shift that the W term ((Delta W) beta_W) explains away (negative = amplifies)
- `spill_pct`: mean |(Delta W) beta_W| on non-injected genes, % of the mean injected shift (residual-SD units)

In [ ]:
ABS_KS = [1, 2, 3, 4, 5, 6, 8, 10]
ABS_LOG2FCS = [1.0, 2.0, 3.0, 5.0]
ABS_CACHE = OUT / 'absorption.csv'
absorb = pd.read_csv(ABS_CACHE) if ABS_CACHE.is_file() else None
if absorb is None or {s: len(c) for s, c in controls.items()} != absorb.groupby('set')['n_controls'].first().to_dict():
    U = universe()
    u_idx = np.array([col[g] for g in U])
    row_of = {n: i for i, n in enumerate(data['names'])}
    di = config.BIAS_COLUMNS.index(DEPTH)
    rows = []
    for split in ('lobo', 'cv'):
        for key, tag, tr, te in fold_spec(split):
            tr_r, te_r = np.array([row_of[n] for n in tr]), np.array([row_of[n] for n in te])
            dep = data['X_raw'][tr_r][:, di]
            d_tr = (dep - dep.mean()) / dep.std()
            Lu_tr = logcpm[tr_r][:, u_idx].astype(np.float64)
            cells_ = {f: np.load(cell_path(split, tag, f)) for f in [0.0] + ABS_LOG2FCS}
            pert = cells_[1.0]['labels']
            test_L = {}
            for f, c in cells_.items():
                counts = Y[te_r].astype(np.float64)
                counts[:, u_idx] = c['y_pert']
                lib_f = lib[te_r] + (c['y_pert'] - Y[te_r][:, u_idx]).sum(1)
                test_L[f] = np.log2(counts / lib_f[:, None] * 1e6 + 1)
            for s, ci in cidx.items():
                fit_f = ruvg_fit(logcpm[tr_r][:, ci].astype(np.float64), max(ABS_KS))
                W_tr = ruvg_transform(fit_f, logcpm[tr_r][:, ci].astype(np.float64))
                W_te = {f: ruvg_transform(fit_f, x[:, ci]) for f, x in test_L.items()}
                for k in ABS_KS:
                    X_tr = np.column_stack([np.ones(len(tr_r)), d_tr, W_tr[:, :k]])
                    beta = np.linalg.lstsq(X_tr, Lu_tr, rcond=None)[0]
                    res_sd = (Lu_tr - X_tr @ beta).std(0) + 1e-9
                    for f in ABS_LOG2FCS:
                        A = (W_te[f][:, :k] - W_te[0.0][:, :k]) @ beta[-k:]
                        dy = test_L[f][:, u_idx] - test_L[0.0][:, u_idx]
                        signal = (np.abs(dy[:, pert]) / res_sd[pert]).mean()
                        rows.append(dict(set=s, split=split, fold=key, k=k, log2fc=f, n_controls=len(ci),
                                         absorbed=A[:, pert].sum() / dy[:, pert].sum(),
                                         spill_pct=100 * (np.abs(A[:, ~pert]) / res_sd[~pert]).mean() / signal))
            print(f'absorption {split}/{tag}', flush=True)
    absorb = pd.DataFrame(rows)
    absorb.to_csv(ABS_CACHE, index=False)
ab = absorb.groupby(['set', 'split', 'k', 'log2fc'])[['absorbed', 'spill_pct']].mean().reset_index()
display(ab[ab['log2fc'] == 3.0].pivot_table(index=['split', 'k'], columns='set', values=['absorbed', 'spill_pct']).round(4))

fig, axes = plt.subplots(2, 2, figsize=(9, 6.5), sharex=True, sharey='row')
for j, split in enumerate(('lobo', 'cv')):
    for s in SETS:
        d = ab[(ab['set'] == s) & (ab['split'] == split) & (ab['log2fc'] == 3.0)]
        axes[0, j].plot(d['k'], 100 * d['absorbed'], 'o-', ms=3, color=SET_COLOR[s], label=s)
        axes[1, j].plot(d['k'], d['spill_pct'], 'o-', ms=3, color=SET_COLOR[s])
    axes[0, j].axhline(0, color='0.5', lw=0.8)
    axes[0, j].set_title(f'{split.upper()}, injected log2FC 3')
    axes[1, j].set_xlabel('k')
    for ax in axes[:, j]:
        ax.axvline(K_MAIN, color='0.5', ls=':', lw=0.8)
axes[0, 0].set_ylabel('Injected signal absorbed by W (%)')
axes[1, 0].set_ylabel('Spillover on non-injected genes\n(% of injected signal)')
axes[0, 1].legend(frameon=False, fontsize=FS_ANNOT_SM)
fig.tight_layout()
fig.savefig(FIG_DIR / 'control_7_absorption.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Comparison of the sets

Two tables, one column per set: all sets at `K_MAIN`, and each set at its own k = the largest k <= `K_MAIN` whose
bootstrap stability (5th percentile) is >= `STAB_MIN` (section 5), since an unstable k penalises a set for a near-tie of
two singular values rather than for its genes. The `CHOSEN` list is written to `splits/control_genes_blood.txt`
(config.PIPELINES['p2']['control_genes']) once; a rerun only checks that it is unchanged, so changing the choice
requires removing the file (and every p2 run fitted on it).

In [ ]:
def compare(k_of):
    rows = {}
    for s in SETS:
        k = k_of[s]
        r = rda[(rda['set'] == s) & (rda['k'] == k)].iloc[0]
        a = ab[(ab['set'] == s) & (ab['split'] == 'lobo') & (ab['k'] == k) & (ab['log2fc'] == 3.0)].iloc[0]
        rows[s] = {
            'k': k,
            'control genes (incl. sex)': len(controls[s]),
            'PanglaoDB blood markers': int((tables[s]['panglao_blood_marker'] != '').sum()),
            'cumulative control variance W_1..k (%)': 100 * kts[s].loc[k, 'var_frac_cum'],
            'stability 5th pct': kts[s].loc[k, 'stab_p5'],
            'R2 known sex ~ W_1..k': kts[s].loc[k, 'sex_r2'],
            'adjusted R2, depth + W_1..k (%)': 100 * r['adj_r2'],
            'batch left, depth + W_1..k (%)': 100 * r['batch_left'],
            'max VIF, depth + W_1..k': r['max_vif'],
            'LOBO absorbed, log2FC 3 (%)': 100 * a['absorbed'],
            'LOBO spillover, log2FC 3 (%)': a['spill_pct'],
            'median phenotype partial R2, set': tables[s]['pheno_r2'].median(),
            'median phenotype partial R2, other C1 genes': others[s].median(),
        }
    comp = pd.DataFrame(rows)
    comp['BIAS10 (p0)'] = np.nan
    comp.loc[['adjusted R2, depth + W_1..k (%)', 'batch left, depth + W_1..k (%)', 'max VIF, depth + W_1..k'], 'BIAS10 (p0)'] = [
        100 * ref_bias['adj_r2'], 100 * ref_bias['batch_left'], ref_bias['max_vif']]
    return comp


k_stable = {s: max([k for k in KS if k <= K_MAIN and kts[s].loc[k, 'stab_p5'] >= STAB_MIN], default=1) for s in SETS}
for name, k_of in ((f'k{K_MAIN}', {s: K_MAIN for s in SETS}), ('k_stable', k_stable)):
    comp = compare(k_of)
    comp.to_csv(OUT / f'comparison_{name}.csv')
    print(name)
    display(comp.round(4))

if CONTROL_LIST.exists():
    assert CONTROL_LIST.read_text().split() == controls[CHOSEN], f'{CONTROL_LIST} differs; remove it and every p2 run to change the set'
else:
    CONTROL_LIST.write_text('\n'.join(controls[CHOSEN]) + '\n')
print(f'{CONTROL_LIST}: {CHOSEN}, {len(controls[CHOSEN])} genes')

## 9. BIAS10 metrics as extra covariates next to W

Which BIAS10 metrics add information W does not already carry? Per set, at its stable k (section 8), on the HC:
- `r2_on_W`: R^2 of the metric regressed on W_1..k (0 = orthogonal to W); `max_abs_r`: largest |r| with a single W_i
- `gain`: adjusted R^2 of HC expression (Y of section 6) gained by adding the metric to depth + W_1..k;
  `batch_drop`: reduction of the batch variance left
- candidates: `r2_on_W` < `ORTHO_R2_MAX` and `gain` >= `GAIN_MIN`; forward selection among them by `gain`, re-evaluated
  after each addition, stopping when no candidate adds `GAIN_MIN`
`moves_with_counts` marks the metrics computed from the counts (they are recomputed after an in-silico injection,
`engine/core/count_metrics.py`); the read-level ones (spliced, gDNA, rRNA, 3' bias) are measured outside the count matrix.

In [ ]:
ORTHO_R2_MAX = 0.2
GAIN_MIN = 0.005
from _FinalModelingMethodImprovement.engine.core.count_metrics import COUNT_METRICS

orth_rows, sel_rows, final_rows = [], [], []
for s in SETS:
    k = k_stable[s]
    Wh = W[s][hc_fit][:, :k]
    base_X = np.hstack([dz, Wh])
    base_r2 = adj_r2(Yc, base_X)
    base_left = adj_r2(Yc, np.hstack([base_X, Bt])) - base_r2
    for j, m in enumerate(config.BIAS_COLUMNS):
        if m == DEPTH:
            continue
        x = Xb[:, [j]]
        X = np.hstack([base_X, x])
        a = adj_r2(Yc, X)
        orth_rows.append(dict(set=s, k=k, metric=m, moves_with_counts=m in COUNT_METRICS, r2_on_W=r2_on(Wh, x[:, 0]),
                              max_abs_r=np.abs([np.corrcoef(Wh[:, i], x[:, 0])[0, 1] for i in range(k)]).max(),
                              gain=a - base_r2, batch_drop=base_left - (adj_r2(Yc, np.hstack([X, Bt])) - a)))
    o = pd.DataFrame([r for r in orth_rows if r['set'] == s]).set_index('metric')
    cand = [m for m in o.index if o.loc[m, 'r2_on_W'] < ORTHO_R2_MAX]
    chosen, X, cur = [], base_X, base_r2
    while cand:
        gains = {m: adj_r2(Yc, np.hstack([X, Xb[:, [config.BIAS_COLUMNS.index(m)]]])) - cur for m in cand}
        m = max(gains, key=gains.get)
        if gains[m] < GAIN_MIN:
            break
        X = np.hstack([X, Xb[:, [config.BIAS_COLUMNS.index(m)]]])
        cur += gains[m]
        chosen.append(m)
        cand.remove(m)
        sel_rows.append(dict(set=s, step=len(chosen), added=m, gain=gains[m], adj_r2=cur))
    for name, Xs in ((f'depth + W_1..{k}', base_X), (f'depth + W_1..{k} + selected ({len(chosen)})', X)):
        a = adj_r2(Yc, Xs)
        final_rows.append(dict(set=s, model=name, n_covariates=Xs.shape[1], adj_r2=a,
                               batch_left=adj_r2(Yc, np.hstack([Xs, Bt])) - a, max_vif=vif(Xs).max(),
                               selected=', '.join(chosen)))
orth = pd.DataFrame(orth_rows)
orth.to_csv(OUT / 'bias_vs_W.csv', index=False)
display(orth.pivot(index='metric', columns='set', values=['r2_on_W', 'gain', 'batch_drop']).round(4))
print('forward selection (r2_on_W < %.2f, gain >= %.3f):' % (ORTHO_R2_MAX, GAIN_MIN))
display(pd.DataFrame(sel_rows).round(4))
final = pd.DataFrame(final_rows + [dict(set='-', model='BIAS10 (p0)', n_covariates=Xb.shape[1], **ref_bias, selected='')])
final.to_csv(OUT / 'bias_selection.csv', index=False)
display(final.round(4))

fig, axes = plt.subplots(1, len(SETS), figsize=(4.6 * len(SETS), 4), sharex=True, sharey=True)
for ax, s in zip(axes, SETS):
    d = orth[orth['set'] == s]
    picked = set(r['added'] for r in sel_rows if r['set'] == s)
    for r in d.itertuples():
        c = SET_COLOR[s] if r.metric in picked else '0.6'
        ax.scatter(r.r2_on_W, 100 * r.gain, s=40, color=c, marker='o' if r.moves_with_counts else 's', zorder=3)
        ax.annotate(r.metric, (r.r2_on_W, 100 * r.gain), fontsize=FS_ANNOT_SM, xytext=(3, 3), textcoords='offset points')
    ax.axvline(ORTHO_R2_MAX, color='k', ls=':'); ax.axhline(100 * GAIN_MIN, color='k', ls=':')
    ax.set_title(f'{s} (k = {k_stable[s]}); coloured = selected')
    ax.set_xlabel('R2 of metric on W_1..k (HC)')
axes[0].set_ylabel('Gain in HC variance explained\nover depth + W (adjusted R2, %)')
axes[0].scatter([], [], marker='o', color='0.6', label='from counts'); axes[0].scatter([], [], marker='s', color='0.6', label='read-level')
axes[0].legend(frameon=False, fontsize=FS_ANNOT_SM)
fig.tight_layout()
fig.savefig(FIG_DIR / 'control_9_bias_vs_W.png', dpi=300, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, len(SETS), figsize=(4.6 * len(SETS), 3.6))
for ax, s in zip(axes, SETS):
    k = k_stable[s]
    cm = np.corrcoef(np.column_stack([W[s][hc_fit][:, :k], data['X_raw'][hc_fit]]).T)[:k, k:]
    im = ax.imshow(cm, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f'{cm[i, j]:.2f}', ha='center', va='center', fontsize=6)
    ax.set_xticks(range(len(config.BIAS_COLUMNS)), config.BIAS_COLUMNS, rotation=45, ha='right', fontsize=FS_ANNOT_SM)
    ax.set_yticks(range(k), [f'W_{i + 1}' for i in range(k)])
    ax.set_title(f'Pearson r, W vs BIAS10 (HC), {s}')
fig.colorbar(im, ax=axes[-1])
fig.tight_layout()
fig.savefig(FIG_DIR / 'control_9_W_bias_corr.png', dpi=300, bbox_inches='tight')
plt.show()